In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo==1.13.3"], 
               capture_output=True)

import importlib
print("dbrepo installed:", importlib.util.find_spec("dbrepo") is not None)

dbrepo installed: True


In [2]:
from dbrepo.RestClient import RestClient
import os
import requests

os.environ["DBREPO_PASSWORD"] = input("Enter your DBRepo password: ")

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password=os.environ.get("DBREPO_PASSWORD")
)

print("Logged in as:", client.whoami())

12534814
Logged in as: 12534814


In [3]:
DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"

tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}

for table in tables:
    full = client.get_table(
        database_id=DATABASE_ID, 
        table_id=table.id
    )
    table_lookup[table.name] = {
        "table_id": table.id,
        "columns": {col.name: col.id for col in full.columns}
    }
    print(f"\n{table.name} (id={table.id})")
    for col in full.columns:
        print(f"  {col.name}: {col.id}")


water_quality_measurement (id=0b680301-8907-4cad-af74-a345489adf5c)
  measurement_id: 5a6ae254-67de-4681-98a9-955e4b3dad8f
  vandens_temp: 364a1444-4a57-47a9-bd49-6c4a5f476dff
  suspend_medziagos: 00aea0b0-5b3a-41fb-a842-e7d44a044d96
  sarmingumas: 39b57ed3-68a1-46c1-b3b0-560a48b58bd0
  deguonis_istirpes: b4362c09-fa4c-4135-81ed-46046a7136f4
  ph: bc605417-78f3-4ddb-9ab9-43e1b74ac60c
  skaidrumas: e21ec1f0-4416-4fcf-876b-c0da94af7c29
  elektr_laidis: 04f34533-2982-413f-be9d-e1558fd226cc
  biochem_deg_suvartojimas: f4617b1f-7191-4054-9e37-ef5538b49984
  amonio_azotas: 56b81d24-de93-4cf6-9e63-a67e2f9f633a
  nitritu_azotas: c61b9a30-789d-4042-ae5d-fd072d2087b5
  nitratu_azotas: 14705582-c4aa-4307-be03-689242e00095
  azotas_mineralinis: b3114f70-e73b-4438-a628-56292d936363
  azotas_bendras: e038b944-8896-4b1b-820e-bb8f1a5495fa
  fosfatu_fosforas: 49818c57-7057-4cf4-bcf3-f284f98cbaf4
  fosforas_bendras: 59657311-ba0a-418c-8f92-e897f25cf2da
  anglingumas: 8d8fc106-8aeb-4bd0-83cd-d13dc91c79a

In [4]:
unit_mappings = [
    # === degree Celsius - SI Digital Framework ===
    ("water_quality_measurement", "vandens_temp",
     "https://si-digital-framework.org/SI/units/CEL"),

    # === metre - SI Digital Framework ===
    ("water_quality_measurement", "skaidrumas",
     "https://si-digital-framework.org/SI/units/m"),

    # === mg/L - QUDT fallback ===
    ("water_quality_measurement", "suspend_medziagos",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "sarmingumas",
     "https://qudt.org/vocab/unit/MilliMOL-PER-L"),
    ("water_quality_measurement", "deguonis_istirpes",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "biochem_deg_suvartojimas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "amonio_azotas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "nitritu_azotas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "nitratu_azotas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "azotas_mineralinis",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "azotas_bendras",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "fosfatu_fosforas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "fosforas_bendras",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "anglingumas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),
    ("water_quality_measurement", "kalcio_karbonatas",
     "https://qudt.org/vocab/unit/MilliGM-PER-L"),

    # === µS/cm - QUDT fallback ===
    ("water_quality_measurement", "elektr_laidis",
     "https://qudt.org/vocab/unit/MicroS-PER-CentiM"),

    # === µg/L - QUDT fallback (chlorophyll + all heavy metals + pollutants) ===
    ("water_quality_measurement", "chlorofilas_a",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "gyvsidabris",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "kadmis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "nikelis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "svinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "varis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "chromas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "vanadis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "aliuminis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "alavas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "arsenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "cinkas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "antracenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "fluorantenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "naftalenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "benz_a_pirenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "benz_b_fluorantenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "benz_k_fluorantenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "benz_ghi_perilenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "inden_123_cd_pirenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_n_nonilfenolis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_n_oktilfenolis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_nonilfenolis_sakot",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_tert_oktilfenolis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "nonilfenoliai",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pentachlorfenolis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "benzenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p12_dichloretanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p123_trichlorbenzenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p124_trichlorbenzenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "heksachlorbutadienas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "trichloretilenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "tetrachlormetanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "dichlormetanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "tetrachloretilenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "trichlormetanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "aldrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "dieldrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "izodrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "endrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "alfa_heksachlorcikloheksanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "beta_heksachlorcikloheksanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "gama_heksachlorcikloheksanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "heksachlorbenzenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pentachlorbenzenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "alfa_endosulfanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "beta_endosulfanas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "o_p_ddt",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p_p_ddd",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p_p_ddt",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p_p_dde",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "simazinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "atrazinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "diuronas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "izoproturonas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "chinoksifenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "aklonifenas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "cibutrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "terbutrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "chlorpyrifosas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "chlorfenvinfosas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "trifluralinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "heptachloras",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "heptachloro_epoksidas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "tributilalavo_katijonas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "di2_etilheksilftalatas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_28",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_47",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_85",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_99",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_100",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_153",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bde_154",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_28",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_52",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_101",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_118",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_138",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_153",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pcb_180",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "pfos",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "dikofolis",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "alachloras",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "bifenoksas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "cipermetrinas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "dichlorvosas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_t_oktilfenolio_dietoksilatas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_t_oktilfenolio_monoetoksilatas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
    ("water_quality_measurement", "p4_t_oktilfenolio_trietoksilatas",
     "https://qudt.org/vocab/unit/MicroGM-PER-L"),
]

print(f"Total unit mappings: {len(unit_mappings)}")

Total unit mappings: 105


In [5]:
BASE_URL = "https://test.dbrepo.tuwien.ac.at"
username = "12534814"
password = os.environ.get("DBREPO_PASSWORD")

success = 0
skipped = 0
errors = 0

for table_name, col_name, unit_uri in unit_mappings:
    if table_name not in table_lookup:
        print(f"⚠ Table not found: {table_name}")
        skipped += 1
        continue
    if col_name not in table_lookup[table_name]["columns"]:
        print(f"⚠ Column not found: {table_name}.{col_name}")
        skipped += 1
        continue

    table_id  = table_lookup[table_name]["table_id"]
    column_id = table_lookup[table_name]["columns"][col_name]
    url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}/column/{column_id}"

    response = requests.put(
        url,
        json={"unit_uri": unit_uri},
        auth=(username, password)
    )

    if response.status_code in [200, 202]:
        print(f"✓ {col_name} → {unit_uri.split('/')[-1]}")
        success += 1
    else:
        print(f"✗ {col_name} → {response.status_code}: {response.text[:80]}")
        errors += 1

print(f"\n=== DONE ===")
print(f"✓ {success} pushed | ⚠ {skipped} skipped | ✗ {errors} errors")

✓ vandens_temp → CEL
✓ skaidrumas → m
✓ suspend_medziagos → MilliGM-PER-L
✓ sarmingumas → MilliMOL-PER-L
✓ deguonis_istirpes → MilliGM-PER-L
✓ biochem_deg_suvartojimas → MilliGM-PER-L
✓ amonio_azotas → MilliGM-PER-L
✓ nitritu_azotas → MilliGM-PER-L
✓ nitratu_azotas → MilliGM-PER-L
✓ azotas_mineralinis → MilliGM-PER-L
✓ azotas_bendras → MilliGM-PER-L
✓ fosfatu_fosforas → MilliGM-PER-L
✓ fosforas_bendras → MilliGM-PER-L
✓ anglingumas → MilliGM-PER-L
✓ kalcio_karbonatas → MilliGM-PER-L
✓ elektr_laidis → MicroS-PER-CentiM
✓ chlorofilas_a → MicroGM-PER-L
✓ gyvsidabris → MicroGM-PER-L
✓ kadmis → MicroGM-PER-L
✓ nikelis → MicroGM-PER-L
✓ svinas → MicroGM-PER-L
✓ varis → MicroGM-PER-L
✓ chromas → MicroGM-PER-L
✓ vanadis → MicroGM-PER-L
✓ aliuminis → MicroGM-PER-L
✓ alavas → MicroGM-PER-L
✓ arsenas → MicroGM-PER-L
✓ cinkas → MicroGM-PER-L
✓ antracenas → MicroGM-PER-L
✓ fluorantenas → MicroGM-PER-L
✓ naftalenas → MicroGM-PER-L
✓ benz_a_pirenas → MicroGM-PER-L
✓ benz_b_fluorantenas → MicroGM-PER-

In [9]:
# Verify by reading the full table and checking column unit_uri
table_id = table_lookup["water_quality_measurement"]["table_id"]
url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}"

r = requests.get(url, auth=(username, password))
data = r.json()

# Print full structure to see what fields are available
import json
print(json.dumps(data, indent=2))

{
  "id": "0b680301-8907-4cad-af74-a345489adf5c",
  "name": "water_quality_measurement",
  "description": "All physicochemical and pollutant measurements for each sampling event",
  "alias": null,
  "identifiers": [],
  "owner": {
    "id": null,
    "username": "12534814",
    "name": null,
    "orcid": null,
    "qualified_name": null,
    "given_name": null,
    "family_name": null
  },
  "columns": [
    {
      "id": "5a6ae254-67de-4681-98a9-955e4b3dad8f",
      "name": "measurement_id",
      "alias": null,
      "size": null,
      "d": null,
      "mean": 1169.5,
      "median": 1169.5,
      "description": null,
      "enums": [],
      "sets": [],
      "database_id": "13457a52-37f9-48d4-a078-6865e8d35981",
      "table_id": "0b680301-8907-4cad-af74-a345489adf5c",
      "ord": 0,
      "internal_name": "measurement_id",
      "index_length": null,
      "length": null,
      "type": "bigint",
      "data_length": null,
      "max_data_length": null,
      "num_rows": null,
  

In [10]:
# Find vandens_temp column and print ALL its fields
data_columns = data["columns"]

for col in data_columns:
    if col["name"] == "vandens_temp":
        print("ALL FIELDS FOR vandens_temp:")
        print(json.dumps(col, indent=2))
        break

ALL FIELDS FOR vandens_temp:
{
  "id": "364a1444-4a57-47a9-bd49-6c4a5f476dff",
  "name": "vandens_temp",
  "alias": null,
  "size": null,
  "d": null,
  "mean": 3.4414,
  "median": 3.4414,
  "description": null,
  "enums": [],
  "sets": [],
  "database_id": "13457a52-37f9-48d4-a078-6865e8d35981",
  "table_id": "0b680301-8907-4cad-af74-a345489adf5c",
  "ord": 1,
  "internal_name": "vandens_temp",
  "index_length": null,
  "length": null,
  "type": "text",
  "data_length": null,
  "max_data_length": null,
  "num_rows": null,
  "val_min": null,
  "val_max": null,
  "std_dev": 0.4986,
  "concept_uri": null,
  "unit_uri": "https://si-digital-framework.org/SI/units/CEL",
  "is_null_allowed": true
}


In [11]:
# Final verification - check unit_uri for key columns
spot_check = ["vandens_temp", "skaidrumas", "ph", "gyvsidabris", "chlorofilas_a", "elektr_laidis"]

print("=== FINAL VERIFICATION ===")
for col in data["columns"]:
    if col["name"] in spot_check:
        uri = col.get("unit_uri", "MISSING")
        status = "✓" if uri else "✗"
        print(f"{status} {col['name']}: {uri}")

=== FINAL VERIFICATION ===
✓ vandens_temp: https://si-digital-framework.org/SI/units/CEL
✗ ph: None
✓ skaidrumas: https://si-digital-framework.org/SI/units/m
✓ elektr_laidis: https://qudt.org/vocab/unit/MicroS-PER-CentiM
✓ chlorofilas_a: https://qudt.org/vocab/unit/MicroGM-PER-L
✓ gyvsidabris: https://qudt.org/vocab/unit/MicroGM-PER-L


In [12]:
# Fix pH - it was missing from the original mapping
col_id = table_lookup["water_quality_measurement"]["columns"]["ph"]
table_id = table_lookup["water_quality_measurement"]["table_id"]
url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}/column/{col_id}"

response = requests.put(
    url,
    json={"unit_uri": "https://qudt.org/vocab/unit/PH"},
    auth=(username, password)
)

if response.status_code in [200, 202]:
    print("✓ ph → https://qudt.org/vocab/unit/PH")
else:
    print(f"✗ Failed: {response.status_code}: {response.text}")

✓ ph → https://qudt.org/vocab/unit/PH


In [13]:
# Re-fetch table and verify ph
r = requests.get(f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}", auth=(username, password))
data = r.json()

for col in data["columns"]:
    if col["name"] == "ph":
        print(f"✓ ph: {col.get('unit_uri')}")
        break

✓ ph: https://qudt.org/vocab/unit/PH
